In [1]:
%pip install tiktoken openai

### 1. Configure OpenAI Client
We'll retrieve your API key from the Colab secrets manager.

In [1]:
from arrow import get
import tiktoken
from openai import OpenAI
#from google.colab import userdata
import getpass

try:
    # Try to get key from secrets
    #api_key = userdata.get('OPENAI_API_KEY')
    api_key = get('OPENAI_API_KEY')
except Exception:
    # Fallback to manual input if secret is missing or timed out
    print('OPENAI_API_KEY secret not found or inaccessible.')
    api_key = getpass.getpass('Please enter your OpenAI API Key: ')

# Initialize OpenAI client
client = OpenAI(api_key=api_key)

OPENAI_API_KEY secret not found or inaccessible.


### 2. Token Counting with Tiktoken
Tiktoken allows you to calculate exactly how many tokens a string contains for a specific model encoding.

In [8]:
def count_tokens(text, model="gpt-4o"):
    try:
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        encoding = tiktoken.get_encoding("cl100k_base")

    tokens = encoding.encode(text)
    return len(tokens)

example_text = "Tiktoken is great for accurate token counting!"
num_tokens = count_tokens(example_text)
print(f"Text: {example_text}")
print(f"Token count: {num_tokens}")

Text: Tiktoken is great for accurate token counting!
Token count: 10


### 3. Using OpenAI LLM
Now we combine token counting with a simple LLM call.

In [11]:
prompt = "Explain the importance of token management in LLMs in one sentence."

# Count tokens before sending
print(f"Prompt tokens: {count_tokens(prompt)}")

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt}]
)

print("\nAI Response:")
print(response.choices[0].message.content)

# Usage metadata from OpenAI
print(f"\nActual tokens used (from API): {response.usage.prompt_tokens}")

Prompt tokens: 14

AI Response:
Token management in LLMs is crucial because it directly affects the efficiency, coherence, and context retention of the model's output by determining how input text is segmented and processed.

Actual tokens used (from API): 21


# Task
Build a secure and efficient RAG-inspired pipeline that integrates PII detection, vector search, and prompt compression. Specifically:
1. Install and configure `faiss-cpu`, `llmlingua`, `presidio-analyzer`, and `presidio-anonymizer`.
2. Use Microsoft Presidio to detect and anonymize Personal Identifiable Information (PII) within user inputs to comply with EU GDPR guidelines.
3. Implement a FAISS vector index to store and retrieve relevant context or guidelines based on similarity.
4. Utilize LLMLingua to compress the retrieved context and system instructions, reducing token consumption while maintaining semantic integrity.
5. Securely interface with an OpenAI LLM using the anonymized and compressed prompts, and provide a summary of how PII protection and token management optimize the workflow.

## Install Libraries

### Subtask:
Install and configure the necessary libraries for vector search, prompt compression, and PII detection.


**Reasoning**:
Install the necessary libraries for vector search, prompt compression, and PII detection, including the required spaCy model.



In [12]:
%pip install faiss-cpu llmlingua presidio-analyzer presidio-anonymizer
!python -m spacy download en_core_web_lg
print('Installation of libraries and spaCy model completed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 115.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 9.5 MB/s eta 0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 43.0.3
    Uninstalling cryptography-43.0.3:
      Successfully uninstalled cryptography-43.0.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.5 which is incompatible.
pyopenssl 24.2.1 requires cryptography<44,>=41.0.5, but you have cryptography 46.0.5 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

**Reasoning**:
Verify the installations by importing the core libraries to ensure all dependencies are correctly configured and accessible in the current environment.



In [13]:
import faiss
import llmlingua
import presidio_analyzer
import presidio_anonymizer
import spacy

# Check if the spacy model is loadable
nlp = spacy.load('en_core_web_lg')

print('Verification Successful: faiss, llmlingua, presidio, and spaCy model are ready.')

Verification Successful: faiss, llmlingua, presidio, and spaCy model are ready.


## PII Detection and Anonymization

### Subtask:
Use Microsoft Presidio to detect and anonymize Personal Identifiable Information (PII) within a sample user input string.


**Reasoning**:
I will implement the PII detection and anonymization logic using Microsoft Presidio to fulfill the subtask instructions.



In [14]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

# 1. Define sample input with PII
sample_input = "My name is John Doe and my email is john.doe@example.com. I live in New York."

# 2. Initialize AnalyzerEngine
# It uses the 'en' nlp engine which defaults to the downloaded spacy model
analyzer = AnalyzerEngine()

# 3. Analyze the text for PII
analysis_results = analyzer.analyze(text=sample_input, entities=["PERSON", "EMAIL_ADDRESS", "LOCATION"], language='en')

# 4. Initialize AnonymizerEngine
anonymizer = AnonymizerEngine()

# 5. Anonymize the detected entities
anonymized_result = anonymizer.anonymize(
    text=sample_input,
    analyzer_results=analysis_results
)

# 6. Print results
print(f"Original Text: {sample_input}")
print(f"Anonymized Text: {anonymized_result.text}")

# Verify entities found
print("\nDetected Entities:")
for res in analysis_results:
    print(f"- Type: {res.entity_type}, Start: {res.start}, End: {res.end}, Score: {res.score}")

Original Text: My name is John Doe and my email is john.doe@example.com. I live in New York.
Anonymized Text: My name is <PERSON> and my email is <EMAIL_ADDRESS>. I live in <LOCATION>.

Detected Entities:
- Type: EMAIL_ADDRESS, Start: 36, End: 56, Score: 1.0
- Type: PERSON, Start: 11, End: 19, Score: 0.85
- Type: LOCATION, Start: 68, End: 76, Score: 0.85


In [15]:
import numpy as np
import faiss

# 1. Define sample context snippets
context_data = [
    "EU GDPR guidelines require strict protection of Personal Identifiable Information (PII).",
    "FAISS is a library for efficient similarity search and clustering of dense vectors.",
    "LLMLingua provides a method to compress long prompts into shorter versions.",
    "Anonymization involves replacing sensitive data with generic placeholders to protect privacy.",
    "Vector databases are essential for building retrieval-augmented generation systems."
]

# 2. Use a simple embedding approach (for demo purposes, using random vectors)
# In a real scenario, you would use OpenAI embeddings or a SentenceTransformer model.
dimensions = 64
np.random.seed(42)
embeddings = np.random.random((len(context_data), dimensions)).astype('float32')

# 3. Initialize FAISS index
index = faiss.IndexFlatL2(dimensions)
index.add(embeddings)

# 4. Define a search function
def retrieve_context(query_text, k=2):
    # Simulate query embedding
    query_vector = np.random.random((1, dimensions)).astype('float32')
    distances, indices = index.search(query_vector, k)
    retrieved = [context_data[i] for i in indices[0]]
    return retrieved

# 5. Test retrieval
query = "How to manage PII under GDPR?"
results = retrieve_context(query)

print(f"Query: {query}")
print("Retrieved Context:")
for i, res in enumerate(results):
    print(f"{i+1}. {res}")

print(f"\nFAISS Index contains {index.ntotal} vectors.")

Query: How to manage PII under GDPR?
Retrieved Context:
1. LLMLingua provides a method to compress long prompts into shorter versions.
2. Anonymization involves replacing sensitive data with generic placeholders to protect privacy.

FAISS Index contains 5 vectors.


In [1]:
%%bash
pip install faiss-cpu presidio-analyzer presidio-anonymizer
python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 1.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


**Reasoning**:
Now that the libraries and the spaCy model have been installed, I will verify the installation by importing the packages and loading the downloaded model to ensure everything is correctly configured.



In [2]:
import faiss
import presidio_analyzer
import presidio_anonymizer
import spacy

# Verify spaCy model loading
try:
    nlp = spacy.load('en_core_web_lg')
    print('Successfully loaded en_core_web_lg model.')
except Exception as e:
    print(f'Error loading model: {e}')

print('Imports and verification completed.')

Successfully loaded en_core_web_lg model.
Imports and verification completed.


## PII Detection and Anonymization

### Subtask:
Implement a PII detection and anonymization workflow using Microsoft Presidio to secure user input.


**Reasoning**:
I will implement a Python script that uses Microsoft Presidio's AnalyzerEngine and AnonymizerEngine to detect and mask PII entities such as names, emails, and phone numbers in a sample string, as specified in the instructions.



In [8]:
from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_anonymizer import AnonymizerEngine

# 1. Instantiate the AnalyzerEngine
analyzer = AnalyzerEngine()

# 2. Add a custom recognizer for phone numbers to handle varied formats
phone_pattern = Pattern(name="phone_number_pattern", regex=r"(\d{3}-\d{4}|\d{3}-\d{3}-\d{4}|\(\d{3}\)\s*\d{3}-\d{4})", score=0.8)
phone_recognizer = PatternRecognizer(supported_entity="PHONE_NUMBER", patterns=[phone_pattern])
analyzer.registry.add_recognizer(phone_recognizer)

# 3. Define sample input
sample_input = "Hello, my name is Alice Smith. You can reach me at alice.smith@example.com or call 555-0199."

# 4. Analyze the text
# We lower score_threshold to 0.4 to capture more potential entities
analysis_results = analyzer.analyze(
    text=sample_input,
    entities=["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER"],
    language='en',
    score_threshold=0.4
)

# 5. Anonymize
anonymizer = AnonymizerEngine()
anonymized_result = anonymizer.anonymize(
    text=sample_input,
    analyzer_results=analysis_results
)

print(f"Original Text: {sample_input}")
print(f"Anonymized Text: {anonymized_result.text}")

print("\nDetected Entities for Verification:")
for result in analysis_results:
    print(f"- Found {result.entity_type} at [{result.start}:{result.end}] with confidence {result.score}")

Original Text: Hello, my name is Alice Smith. You can reach me at alice.smith@example.com or call 555-0199.
Anonymized Text: Hello, my name is <PERSON>. You can reach me at <EMAIL_ADDRESS> or call <PHONE_NUMBER>.

Detected Entities for Verification:
- Found EMAIL_ADDRESS at [51:74] with confidence 1.0
- Found PERSON at [18:29] with confidence 0.85
- Found PHONE_NUMBER at [83:91] with confidence 0.8


**Reasoning**:
I will implement a FAISS vector index using spaCy 'en_core_web_lg' embeddings to store knowledge base strings and create a retrieval function to find the most similar context snippets.



In [4]:
import numpy as np
import faiss
import spacy

# 1. Define knowledge base context guidelines
knowledge_base = [
    "GDPR Rule: Personal data must be processed lawfully, fairly, and transparently.",
    "Security Protocol: Use multi-factor authentication for all administrative access.",
    "GDPR Rule: Individuals have the right to be forgotten and request data deletion.",
    "Security Protocol: Encrypt all sensitive data at rest and in transit using AES-256.",
    "EU Regulation: Organizations must report data breaches within 72 hours of discovery."
]

# 2. Load the 'en_core_web_lg' spaCy model (already verified in previous steps)
nlp = spacy.load('en_core_web_lg')

# 3. Generate high-dimensional embeddings using spaCy .vector
# spaCy 'en_core_web_lg' vectors usually have 300 dimensions
kb_vectors = np.array([nlp(text).vector for text in knowledge_base]).astype('float32')

# 4. Initialize FAISS IndexFlatL2
dimension = kb_vectors.shape[1]
index = faiss.IndexFlatL2(dimension)

# 5. Add embeddings to the index
index.add(kb_vectors)

# 6. Create a retrieval function
def retrieve_knowledge(query, k=2):
    # Convert query to spaCy vector
    query_vector = nlp(query).vector.reshape(1, -1).astype('float32')
    # Search FAISS index
    distances, indices = index.search(query_vector, k)
    # Return the corresponding strings
    return [knowledge_base[i] for i in indices[0]]

# 7. Test the function
test_query = "How should I handle a data leak?"
retrieved_snippets = retrieve_knowledge(test_query)

print(f"Query: {test_query}")
print("Top Retrieved Snippets:")
for i, snippet in enumerate(retrieved_snippets, 1):
    print(f"{i}. {snippet}")

print(f"\nFAISS index dimensionality: {dimension}")
print(f"Total vectors in index: {index.ntotal}")

Query: How should I handle a data leak?
Top Retrieved Snippets:
1. GDPR Rule: Individuals have the right to be forgotten and request data deletion.
2. Security Protocol: Encrypt all sensitive data at rest and in transit using AES-256.

FAISS index dimensionality: 300
Total vectors in index: 5


In [7]:
from google.colab import userdata
from openai import OpenAI

# 1. Re-initialize the OpenAI client
try:
    api_key = userdata.get('OPENAI_API_KEY')
    client = OpenAI(api_key=api_key)
except Exception as e:
    print(f'Error initializing OpenAI client: {e}')

# 2. Retrieve the anonymized user text
anonymized_query = anonymized_result.text

# 3. Retrieve relevant context from FAISS
retrieved_context = retrieve_knowledge(anonymized_query, k=2)
context_string = '\n'.join(retrieved_context)

# 4. Construct the prompt
prompt = f"""Context Guidelines:
{context_string}

User Request (Anonymized for Privacy):
{anonymized_query}

Task: Based on the context guidelines provided above, please provide advice to the user while maintaining their privacy."""

# 5. Call OpenAI
try:
    completion = client.chat.completions.create(
        model='gpt-4o',
        messages=[
            {'role': 'system', 'content': 'You are a helpful assistant specialized in GDPR compliance.'},
            {'role': 'user', 'content': prompt}
        ]
    )

    llm_output = completion.choices[0].message.content
    print('--- Secure LLM Response (Anonymized) ---')
    print(llm_output)

    # 6. Final Step: Local De-anonymization (Restoring PII locally)
    # We map placeholders back to original values for the final user view
    final_text = llm_output
    for item in anonymized_result.items:
        placeholder = f"<{item.entity_type}>"
        # In a real app, you would map specific instances correctly
        # For this demo, we replace the tags with the original substring
        original_value = sample_input[item.start:item.end]
        final_text = final_text.replace(placeholder, original_value)

    print('\n--- Final Reconstructed Text (Local Only) ---')
    print(final_text)

except Exception as e:
    print(f'An error occurred: {e}')

--- Secure LLM Response (Anonymized) ---
Hello,

Thank you for reaching out. Under the GDPR, you have several rights concerning your personal data, which include the right to be forgotten and the right to request data deletion. Here is some advice based on these rights:

1. **Right to be Forgotten**: If you wish for any organization to delete your personal data, you can make a formal request to them to have your information erased. This is applicable when the data is no longer necessary for the purpose it was collected, or if you withdraw your consent for its use.

2. **Data Deletion Request**: Identify which organizations hold your personal data and submit a request for deletion. Ensure your request clearly states that you are exercising your right under the GDPR to have your personal data erased.

3. **Contacting Organizations**: Use secure methods to communicate your request to the organizations holding your data. Avoid sharing sensitive personal information, such as your name, emai